In [1]:
import time
import numpy as np
import pandas as pd
from datetime import datetime
from tts_data_utils.core.data_frame import TtsDataFrame
from copy import deepcopy

In [2]:
class TestDataFrame(TtsDataFrame):
    SCHEMA = [
        ('label', str),
        ('value', (int, float, str)),
        ('units', str),
        ('time', datetime),
    ]

    TIME_FORMATS = {
        'time': '%Y-%jT%H:%M:%S.%f'
    }
    DEFAULT_TIME_LABEL = 'time'

    LABEL_COL = 'label'

    VALUE_COL = 'value'

    LABEL_COLUMN = 'label'

    DEFAULT_DIRECTION = 'horizontal'

    def __init__(self, *args, name=None, metadata=None, **kwargs):
        # Provide a sensible default name similar to EvrContainer
        if name is None:
            name = "Test DataFrame"
        super().__init__(*args, name=name, metadata=metadata, **kwargs)



## 1. Valid data — coerce + validate

In [3]:
df = TestDataFrame(
    data={
        "label":  ["temp",   "speed", "pressure", "speed"],
        "value":  [42,       "slow", 3.14,       "fast"],
        "units":  ["C",      "knots", "psi",      "knots"],
        "time":   ["2024-001T00:00:00.000000",
                   "2024-001T00:00:00.000000",
                   "2024-001T01:00:00.000000",
                   "2024-001T02:00:00.000000"],
    },
    coerce=True,
    validate=True,
)
display(df)
df.dtypes

,label,value,units,time
0,temp,42,C,2024-01-01 00:00:00
1,speed,slow,knots,2024-01-01 00:00:00
2,pressure,3.14,psi,2024-01-01 01:00:00
3,speed,fast,knots,2024-01-01 02:00:00


label               str
value            object
units               str
time     datetime64[us]
dtype: object

## 2. `.valid` property

In [4]:
df.valid

True

## 3. `'1.1'` string in int/float/str column — should land as float

In [5]:
df2 = TestDataFrame(
    data={
        "label":  ["x"],
        "value":  ["1.1"],
        "units":  ["m"],
        "time":   ["2024-001T00:00:00.000000"],
    },
    coerce=True,
    validate=True,
)
display(df2)
print("value[0]:", df2["value"].iloc[0], type(df2["value"].iloc[0]))

,label,value,units,time
0,x,1.1,m,2024-01-01


value[0]: 1.1 <class 'str'>


## 4. Validation failure — int in `units` column

In [6]:
try:
    df3 = TestDataFrame(
        data={
            "label":  ["y"],
            "value":  [1],
            "units":  [999],
            "time":   ["2024-001T00:00:00.000000"],
        },
        coerce=False,
        validate=True,
    )
except Exception as e:
    print("Caught expected error:", e)

Caught expected error: Column 'units' has values with invalid type for schema <class 'str'>. Example bad indices: [0]


## 5. Missing column

In [7]:
try:
    df4 = TestDataFrame(
        data={"label": ["a"], "value": [1], "units": ["m"]},
        coerce=True,
        validate=True,
    )
except Exception as e:
    print("Caught expected error:", e)

Caught expected error: Missing expected schema columns: ['time']; present columns: ['label', 'value', 'units']


## 6. Stress test — 10 M rows, 100 labels

In [8]:
N_ROWS        = 10_000_000
N_LABELS      = 100
TIME_START    = datetime(2024, 1, 1)
TIME_STEP_SEC = 1

label_names = [f"sensor_{i:03d}" for i in range(N_LABELS)]
units_map   = {l: f"unit_{i % 10}" for i, l in enumerate(label_names)}

rng        = np.random.default_rng(42)
base_times = pd.date_range(TIME_START, periods=N_ROWS // N_LABELS, freq=f"{TIME_STEP_SEC}s")
# Each label gets exactly one row per timestamp; shuffle rows for realism
labels = np.repeat(label_names, N_ROWS // N_LABELS)
times  = np.tile(base_times, N_LABELS)
idx    = rng.permutation(N_ROWS)
labels, times = labels[idx], times[idx]
units  = np.array([units_map[l] for l in labels])
values = rng.random(N_ROWS)

t0 = time.perf_counter()
stress_df = TestDataFrame(
    data={"label": labels, "value": values, "units": units, "time": times},
    coerce=False,
    validate=False,
)
print(f"Built in {time.perf_counter() - t0:.2f}s")

# Thin out sensor_001 (every 3rd sample) and sensor_002 (every 5th sample)
# so they have gaps relative to the other channels.
drop_001 = stress_df.index[stress_df['label'] == 'sensor_001'][::3]
drop_002 = stress_df.index[stress_df['label'] == 'sensor_002'][::5]
stress_df = stress_df.drop(index=drop_001.union(drop_002))

stress_df.info(memory_usage='deep')

Built in 0.55s
<class '__main__.TestDataFrame'>
Index: 9946666 entries, 0 to 9999999
Data columns (total 4 columns):
 #   Column  Dtype         
---  ------  -----         
 0   label   str           
 1   value   float64       
 2   units   str           
 3   time    datetime64[us]
dtypes: datetime64[us](1), float64(1), str(2)
memory usage: 1.3 GB


## 7. Pivot to wide form (with cache)

In [9]:
t1 = time.perf_counter()
wide = stress_df.pivot_table()
print(f"First pivot:  {time.perf_counter() - t1:.2f}s  shape={wide.shape}")

t2 = time.perf_counter()
wide = stress_df.pivot_table()
print(f"Cached pivot: {time.perf_counter() - t2:.4f}s  shape={wide.shape}")
stress_df._pivot_cache.info(memory_usage='deep')

First pivot:  2.69s  shape=(100000, 100)
Cached pivot: 0.2456s  shape=(100000, 100)
<class '__main__.TestDataFrame'>
DatetimeIndex: 100000 entries, 2024-01-01 00:00:00 to 2024-01-02 03:46:39
Data columns (total 100 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   sensor_000  100000 non-null  float64
 1   sensor_001  66666 non-null   float64
 2   sensor_002  80000 non-null   float64
 3   sensor_003  100000 non-null  float64
 4   sensor_004  100000 non-null  float64
 5   sensor_005  100000 non-null  float64
 6   sensor_006  100000 non-null  float64
 7   sensor_007  100000 non-null  float64
 8   sensor_008  100000 non-null  float64
 9   sensor_009  100000 non-null  float64
 10  sensor_010  100000 non-null  float64
 11  sensor_011  100000 non-null  float64
 12  sensor_012  100000 non-null  float64
 13  sensor_013  100000 non-null  float64
 14  sensor_014  100000 non-null  float64
 15  sensor_015  100000 non-null  float64
 16  sensor_016  10

## 8. derive_values — sensor_001 * 2 - abs(sensor_002)

In [10]:
t3 = time.perf_counter()
derived = stress_df.derive_values('new_channel = sensor_001 * 2 - abs(sensor_002)')
print(f"derive_values: {time.perf_counter() - t3:.2f}s")
display(derived.head(10))

derive_values: 2.24s


,time,label,value
0,2024-01-01 00:00:00,new_channel,-0.264575
1,2024-01-01 00:00:01,new_channel,0.044271
2,2024-01-01 00:00:02,new_channel,0.142873
3,2024-01-01 00:00:03,new_channel,1.007349
4,2024-01-01 00:00:04,new_channel,0.862592
5,2024-01-01 00:00:05,new_channel,1.356036
6,2024-01-01 00:00:06,new_channel,0.486827
7,2024-01-01 00:00:08,new_channel,0.079391
8,2024-01-01 00:00:09,new_channel,0.405274
9,2024-01-01 00:00:10,new_channel,0.644469


In [19]:
stress_df.at_times_where('sensor_001 > 0.999').query('label == "sensor_002" | label == "sensor_001"').pivot_table()

label,sensor_001,sensor_002
time,,
2024-01-01 00:18:23,0.999752,0.072510
2024-01-01 01:30:28,0.999258,0.275823
2024-01-01 01:45:25,0.999836,0.385703
2024-01-01 02:32:32,0.999797,NaN
2024-01-01 02:53:46,0.999787,0.254556
...,...,...
2024-01-02 02:48:01,0.999788,0.787644
2024-01-02 03:01:32,0.999446,NaN
2024-01-02 03:02:03,0.999750,NaN
